In [ ]:
from google.colab import files
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Please upload train_df.csv")
uploaded_train = files.upload()
train_filename = next(iter(uploaded_train))
train_df = pd.read_csv(train_filename)
display(train_df.head(3))

print("\nPlease upload test_df.csv")
uploaded_test = files.upload()
test_filename = next(iter(uploaded_test))
test_df = pd.read_csv(test_filename)

TARGET = "readmitted"

Please upload train_df.csv


TypeError: 'NoneType' object is not subscriptable

In [ ]:
train_df = train_df.drop_duplicates().copy()

num_cols = train_df.select_dtypes(include="number").columns.tolist()
cat_cols = train_df.select_dtypes(include="object").columns.tolist()
if TARGET in num_cols:
  num_cols.remove(TARGET)

for col in num_cols:
  train_df[col] = train_df[col].fillna(train_df[col].median())
for col in cat_cols:
  if not train_df[col].mode().empty:
    train_df[col] = train_df[col].fillna(train_df[col].mode()[0])

In [ ]:
y_full = train_df[TARGET].copy()
train_features = train_df.drop(columns=[TARGET]).copy()
test_features = test_df.copy()

train_features["__is_train__"] = 1
test_features["__is_train__"] = 0

combined = pd.concat([train_features, test_features], axis=0, ignore_index=True)
cat_features = combined.select_dtypes(include="object").columns.tolist()
combined_encoded = pd.get_dummies(combined, columns=cat_features, drop_first=True)

X_full = combined_encoded[combined_encoded["__is_train__"] == 1].drop(
    columns=["__is_train__"]
)
X_submission = combined_encoded[combined_encoded["__is_train__"] == 0].drop(
    columns=["__is_train__"]
)
display(X_full.head(3))

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.20, random_state=42, stratify=y_full
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)


model = LogisticRegression(penalty="l2", C=1.0, class_weight="balanced", max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

In [ ]:
y_pred = model.predict(X_val_scaled)
y_prob = model.predict_proba(X_val_scaled)[:, 1]

auc = roc_auc_score(y_val, y_prob)
tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()

print(f"\n--- Model Evaluation ---")
print(f"ROC-AUC Score: {auc:.4f}\n")
print(f"True Negatives : {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives : {tp}\n")


In [ ]:
print("--- Clinical Cost Discussion ---")
print(
    "False Negatives (FN): High Cost. A high-risk patient is falsely identified"
    " as safe, sending them home without post-discharge care and increasing"
    " risk of severe complications or emergency readmission."
)
print(
    "False Positives (FP): Low Cost. A low-risk patient is flagged as"
    " high-risk, leading to unnecessary follow-up calls or care coordination"
    " expenses, but ensuring patient safety."
)
print(
    "Conclusion: In clinical settings, minimizing False Negatives (maximizing"
    " Recall) is prioritized over minimizing False Positives.\n"
)

In [ ]:
X_submission_scaled = scaler.transform(X_submission)
submission_pred = model.predict(X_submission_scaled)
submission_prob = model.predict_proba(X_submission_scaled)[:, 1]

submission = pd.DataFrame({
    "patient_id": range(1, len(test_df) + 1),
    "readmitted_prediction": submission_pred,
    "readmission_probability": submission_prob.round(4),
})

submission.to_csv("submission.csv", index=False)
files.download("submission.csv")
print("submission.csv generated and downloaded!")

display(submission.head(5))